In [98]:
import argparse
import json
import os
from pathlib import Path
from typing import Sequence, Optional, Dict, Tuple

import matplotlib.pyplot as plt
import numpy as np
from numpy import ndarray
from tqdm import tqdm

from keras_models import generate_ncp_model
import tensorflow as tf

from typing import Tuple, Optional

import PIL.Image
import numpy as np
import tensorflow as tf
from numpy import ndarray
from tensorflow.python.keras.models import Functional


In [99]:
!export TF_CPP_MIN_LOG_LEVEL=2

In [117]:
def generate_hidden_list(model: Functional, return_numpy: bool = True):
    print(model.input_shape)
    constructor = np.zeros if return_numpy else tf.zeros
    hiddens = []
    if len(model.input_shape)==1:
        lool = model.input_shape[0][1:]
    else:
        lool = model.input_shape[1:]
    print(lool)
    for input_shape in lool:  # ignore 1st output, as is this control output
        hidden = []
        for i, shape in enumerate(input_shape):
            if shape is None:
                if i == 0:  # batch dim
                    hidden.append(1)
                    continue
                elif i == 1:  # seq len dim
                    hidden.append(0)
                    continue
                else:
                    print("Unable to infer hidden state shape. Leaving as none")
            hidden.append(shape)
        hiddens.append(constructor(hidden))
    return hiddens

In [118]:
def load_image(img_path: str, img_shape: Tuple[int, int, int], reverse_channels: bool = True) -> Optional[ndarray]:
    img = PIL.Image.open(img_path)
    if img is not None:
        # image shape is height, width, PIL takes width, height
        resized = img.resize(img_shape[:2][::-1], PIL.Image.BILINEAR)
        img_numpy = tf.keras.preprocessing.image.img_to_array(resized).astype(np.uint8)
        if reverse_channels:
            # reverse channels of image to match training
            img_numpy = img_numpy[..., ::-1]

        # add batch dim
        img_numpy = np.expand_dims(img_numpy, axis=0)
        return img_numpy
    else:
        return None


def image_dir_generator(data_path: str, image_shape: Tuple[int, int, int], reverse_channels: bool = False):
    
    contents = os.listdir(data_path)
    contents = [os.path.join(data_path, c) for c in contents if 'png' in c]
    contents.sort()
    for path in contents:
        img = load_image(img_path=path, img_shape=image_shape, reverse_channels=reverse_channels)
        if img is not None:
            yield img

In [120]:
reverse_channels = False
sequence_path = "../../fly_to_target_dataset/dataset/1"

IMAGE_SHAPE = (144, 256, 3)
IMAGE_SHAPE_CV = (IMAGE_SHAPE[1], IMAGE_SHAPE[0])

batch_size = None
seq_len = 64
augmentation_params = None
single_step = True
no_norm_layer = False
DROPOUT = 0.1

DEFAULT_NCP_SEED = 22222

mymodel = generate_ncp_model(seq_len, IMAGE_SHAPE, augmentation_params, batch_size, DEFAULT_NCP_SEED, single_step, no_norm_layer)

mymodel.load_weights('../saved_models/fine_tuned_woscheduler_seed22222_lr0.0001_150traj.h5')

hiddens = generate_hidden_list(model=mymodel, return_numpy=True)
print("hidden", hiddens)

all_hiddens = []  # list of list of arrays with shape num_timesteps x num_hiddens x hidden_dim
for i, img in tqdm(enumerate(image_dir_generator(sequence_path, IMAGE_SHAPE, reverse_channels))):
    all_hiddens.append(hiddens)
    out = mymodel.predict([img, *hiddens])
    hiddens = out[1:]  # list num_hidden long, each el is hidden_dim,

# print("all_hiddens", all_hiddens)
# flatten batch dim
all_hiddens = [[np.squeeze(hid, axis=0) for hid in step_hid] for step_hid in all_hiddens]
# print("all_hiddens", all_hiddens)
# crete list with same shape as hidden vectors where contents are lipschitz values of each dimension
lip = [np.zeros_like(h) for h in all_hiddens[0]]
for i in range(len(all_hiddens) - 1):
    current_hiddens = all_hiddens[i]
    next_hiddens = all_hiddens[i + 1]
    diff = [np.abs(n - c) for n, c in zip(next_hiddens, current_hiddens)]
    lip = [np.maximum(l, d) for l, d in zip(lip, diff)]
print(lip)


[(None, 144, 256, 3), (None, 34)]
[(None, 34)]
hidden [array([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0.]])]


832it [00:28, 29.65it/s]

[array([0.29475325, 0.29018787, 0.24389219, 0.27417505, 0.45551652,
       0.17504659, 0.44344532, 0.24757184, 0.48206246, 0.15962578,
       0.82027656, 0.13916595, 0.08889347, 0.79018986, 0.74798185,
       0.25715578, 0.62777293, 0.58725464, 0.67033947, 0.79038382,
       0.72523761, 0.66524971, 0.50155616, 0.68377995, 1.24804032,
       0.56179285, 0.77221298, 1.00452602, 0.2956436 , 0.33006525,
       0.91824853, 0.46289095, 0.87805057, 0.72570515])]
